# Hierarchical Residual Bias Audit

Stage 1 of the hierarchical modeling track. This notebook audits whether the current Huber + additive Hilbert residual model leaves well-level residual structure that could justify contextual partial pooling.

## 1. Setup

In [1]:
"Configure imports, paths, and deterministic audit settings."

from datetime import datetime
from pathlib import Path
import json
import math
import shutil

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "context").exists():
    PROJECT_ROOT = next(parent for parent in PROJECT_ROOT.parents if (parent / "context").exists())

DATE_STAMP = datetime.now().strftime("%Y-%m-%d")
RUN_ID = f"{DATE_STAMP}__hierarchical_residual_bias_audit"

ARTIFACT_TABLES = PROJECT_ROOT / "artifacts" / "tables"
ARTIFACT_REPORTS = PROJECT_ROOT / "artifacts" / "reports"
ARTIFACT_LOGS = PROJECT_ROOT / "artifacts" / "logs" / "runs" / RUN_ID
ARTIFACT_FIGURES = PROJECT_ROOT / "artifacts" / "figures"

for path in [ARTIFACT_TABLES, ARTIFACT_REPORTS, ARTIFACT_LOGS, ARTIFACT_FIGURES]:
    path.mkdir(parents=True, exist_ok=True)

RESIDUAL_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_residual_sample__2026-06-22.csv"
FOLD_METRICS_PATH = ARTIFACT_TABLES / "model__hsgp_residual_on_gam_huber_fold_metrics__2026-06-22.csv"
TARGET_CANDIDATE = "huber_plus_hsgp_dps_m8"
PUBLIC_BASELINE_RMSE = 14.514
LOCAL_BASELINE_RMSE = 14.3586

print(f"project_root={PROJECT_ROOT}")
print(f"run_id={RUN_ID}")

project_root=C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling
run_id=2026-06-22__hierarchical_residual_bias_audit


## 2. Load Residuals

In [2]:
"Load the promoted candidate residual sample from the previous out-of-fold Huber + Hilbert run."

residuals_all = pd.read_csv(RESIDUAL_PATH)
residuals = residuals_all.loc[residuals_all["candidate"].eq(TARGET_CANDIDATE)].copy()

numeric_cols = [
    "fold",
    "row_idx",
    "distance_from_boundary_md",
    "eval_progress",
    "abs_projected_delta_10",
    "y_true",
    "mean_pred_tvt",
    "y_pred",
    "residual",
    "residual_correction",
]
for col in numeric_cols:
    residuals[col] = pd.to_numeric(residuals[col], errors="coerce")

residuals["abs_residual"] = residuals["residual"].abs()
residuals["squared_residual"] = residuals["residual"] ** 2
residuals["distance_log1p"] = np.log1p(residuals["distance_from_boundary_md"].clip(lower=0))

print(f"candidate={TARGET_CANDIDATE}")
print(f"rows={len(residuals):,}")
print(f"wells={residuals['wellname'].nunique():,}")
print(residuals.groupby("fold")["wellname"].nunique().rename("wells_by_fold").to_string())

candidate=huber_plus_hsgp_dps_m8


rows=5,400
wells=773
fold
1    258
2    258
3    257


## 3. Well And Fold Residual Structure

In [3]:
"Summarize residual magnitude, signed bias, and row-level correction behavior by held-out well."

def rmse(values):
    arr = np.asarray(values, dtype=float)
    return float(np.sqrt(np.nanmean(arr ** 2)))

well_summary = (
    residuals.groupby(["fold", "wellname"], as_index=False)
    .agg(
        n_residual_rows=("residual", "size"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        residual_std=("residual", "std"),
        mae=("abs_residual", "mean"),
        p90_abs_residual=("abs_residual", lambda s: float(np.nanpercentile(s, 90))),
        rmse=("residual", rmse),
        mean_residual_correction=("residual_correction", "mean"),
        residual_correction_std=("residual_correction", "std"),
        max_eval_progress=("eval_progress", "max"),
        mean_eval_progress=("eval_progress", "mean"),
        max_distance_from_boundary_md=("distance_from_boundary_md", "max"),
        mean_abs_projected_delta_10=("abs_projected_delta_10", "mean"),
        max_abs_projected_delta_10=("abs_projected_delta_10", "max"),
    )
)
well_summary["abs_mean_residual"] = well_summary["mean_residual"].abs()
well_summary["hierarchy_priority_score"] = (
    well_summary["abs_mean_residual"].rank(pct=True)
    + well_summary["rmse"].rank(pct=True)
    + well_summary["p90_abs_residual"].rank(pct=True)
) / 3

fold_summary = (
    residuals.groupby("fold", as_index=False)
    .agg(
        n_rows=("residual", "size"),
        n_wells=("wellname", "nunique"),
        mean_residual=("residual", "mean"),
        median_residual=("residual", "median"),
        residual_std=("residual", "std"),
        mae=("abs_residual", "mean"),
        p90_abs_residual=("abs_residual", lambda s: float(np.nanpercentile(s, 90))),
        rmse=("residual", rmse),
        mean_residual_correction=("residual_correction", "mean"),
        residual_correction_std=("residual_correction", "std"),
    )
)
fold_summary["abs_mean_residual"] = fold_summary["mean_residual"].abs()

well_summary.sort_values("hierarchy_priority_score", ascending=False).head(10)

,fold,wellname,n_residual_rows,mean_residual,median_residual,residual_std,mae,p90_abs_residual,rmse,mean_residual_correction,residual_correction_std,max_eval_progress,mean_eval_progress,max_distance_from_boundary_md,mean_abs_projected_delta_10,max_abs_projected_delta_10,abs_mean_residual,hierarchy_priority_score
78,1,1b1eba53,7,51.939189,66.200182,33.486032,53.953115,82.603796,60.488064,0.677507,0.560868,0.877525,0.445914,4085.0,73.823492,145.244444,51.939189,0.999138
383,2,86454a6f,7,56.453739,71.785194,24.723509,56.453739,74.682318,60.917607,-0.898806,1.582838,0.913600,0.481961,7276.0,51.184762,97.013333,56.453739,0.998706
321,2,708caea9,7,50.638164,50.854806,21.500329,50.638164,74.516898,54.410018,-0.594493,0.889051,0.963953,0.531279,6312.0,38.658730,70.133333,50.638164,0.996550
551,3,ba48188d,7,-41.955380,-41.484441,34.612833,41.955380,82.318232,52.793488,-0.712510,0.379343,1.000000,0.568239,4166.0,15.784762,27.773333,41.955380,0.995688
145,1,2fd68f7b,7,49.971579,51.717294,16.584659,49.971579,61.838008,52.277305,0.445093,0.532834,0.992599,0.561037,4695.0,47.184762,83.466667,49.971579,0.993963
743,3,f6d009f4,7,36.445223,36.952915,34.483681,37.527737,78.011057,48.451044,5.550139,1.565659,1.000000,0.568237,6715.0,1072.760159,1887.661111,36.445223,0.991807
364,2,7e721392,7,-46.212939,-52.122400,15.043529,46.212939,58.772105,48.266073,-0.166314,0.747957,0.985582,0.553864,6085.0,64.600000,114.938889,46.212939,0.990944
285,2,5f4d2a52,7,39.273101,49.759239,28.399091,39.482330,65.665447,47.261711,-0.120322,0.457566,0.971095,0.533308,5074.0,46.450000,84.566667,39.273101,0.990944
769,3,fef8af96,7,-37.410840,-27.326188,24.822397,37.410840,66.436793,43.905589,0.430364,0.551460,1.000000,0.568217,3826.0,43.488571,76.520000,37.410840,0.988788
391,2,896d15b9,7,38.498001,49.535547,24.689968,38.498001,59.803402,44.772823,0.307892,0.592266,0.949509,0.513791,4063.0,70.856667,130.918889,38.498001,0.988788


## 4. Test-Safe Well Descriptors

In [4]:
"Build held-out well descriptors from train horizontal files and residual coordinates."

train_dir = PROJECT_ROOT / "data" / "raw" / "train"
well_rows = []
for wellname in sorted(well_summary["wellname"].unique()):
    path = train_dir / f"{wellname}__horizontal_well.csv"
    if not path.exists():
        continue
    df = pd.read_csv(path)
    tvt_input = pd.to_numeric(df.get("TVT_input"), errors="coerce") if "TVT_input" in df else pd.Series(np.nan, index=df.index)
    gr = pd.to_numeric(df.get("GR"), errors="coerce") if "GR" in df else pd.Series(np.nan, index=df.index)
    md = pd.to_numeric(df.get("MD"), errors="coerce") if "MD" in df else pd.Series(np.nan, index=df.index)
    x = pd.to_numeric(df.get("X"), errors="coerce") if "X" in df else pd.Series(np.nan, index=df.index)
    y = pd.to_numeric(df.get("Y"), errors="coerce") if "Y" in df else pd.Series(np.nan, index=df.index)
    z = pd.to_numeric(df.get("Z"), errors="coerce") if "Z" in df else pd.Series(np.nan, index=df.index)
    known_mask = tvt_input.notna()
    known_tvt = tvt_input.loc[known_mask]
    recent_known = known_tvt.tail(50)
    recent_slope = np.nan
    if len(recent_known) >= 2:
        recent_md = md.loc[recent_known.index]
        md_delta = float(recent_md.iloc[-1] - recent_md.iloc[0])
        if abs(md_delta) > 1e-9:
            recent_slope = float((recent_known.iloc[-1] - recent_known.iloc[0]) / md_delta)
    well_rows.append(
        {
            "wellname": wellname,
            "source_rows": int(len(df)),
            "md_span": float(md.max() - md.min()) if md.notna().any() else np.nan,
            "z_span": float(z.max() - z.min()) if z.notna().any() else np.nan,
            "xy_span": float(np.hypot(x.max() - x.min(), y.max() - y.min())) if x.notna().any() and y.notna().any() else np.nan,
            "gr_missing_rate": float(gr.isna().mean()),
            "gr_mean": float(gr.mean()) if gr.notna().any() else np.nan,
            "gr_std": float(gr.std()) if gr.notna().sum() > 1 else np.nan,
            "known_tvt_rows": int(known_mask.sum()),
            "known_tvt_fraction": float(known_mask.mean()),
            "last_known_tvt_input": float(known_tvt.iloc[-1]) if len(known_tvt) else np.nan,
            "recent_tvt_slope_last_50": recent_slope,
        }
    )

well_descriptors = pd.DataFrame(well_rows)
well_audit = well_summary.merge(well_descriptors, on="wellname", how="left")

residual_coordinate_summary = (
    residuals.groupby("wellname", as_index=False)
    .agg(
        residual_rows=("residual", "size"),
        max_distance_log1p=("distance_log1p", "max"),
        mean_distance_log1p=("distance_log1p", "mean"),
        mean_eval_progress_row=("eval_progress", "mean"),
        mean_abs_projected_delta_10_row=("abs_projected_delta_10", "mean"),
    )
)
well_audit = well_audit.merge(residual_coordinate_summary, on="wellname", how="left")
well_audit.head()

,fold,wellname,n_residual_rows,mean_residual,median_residual,residual_std,mae,p90_abs_residual,rmse,mean_residual_correction,...,gr_std,known_tvt_rows,known_tvt_fraction,last_known_tvt_input,recent_tvt_slope_last_50,residual_rows,max_distance_log1p,mean_distance_log1p,mean_eval_progress_row,mean_abs_projected_delta_10_row
0,1,000d7d20,7,-3.284996,-2.633730,5.729891,4.985987,10.815113,6.239601,-0.262250,...,18.627493,1442,0.273210,11747.37,0.016327,7,8.105308,6.451065,0.431514,18.398413
1,1,00bbac68,7,-9.038254,-2.812257,15.909423,11.127706,31.089188,17.281240,0.388238,...,17.157431,1545,0.204392,12223.54,0.013061,7,8.563313,6.837554,0.432610,43.371429
2,1,00e12e8b,7,-6.779790,-8.861121,6.834350,8.190778,13.340101,9.273687,0.677843,...,33.321410,2083,0.326284,11604.82,0.041633,7,8.228177,6.962478,0.438738,77.600159
3,1,015fe0d2,7,11.398129,15.112103,11.650720,12.444902,23.981681,15.692843,0.645877,...,19.981056,1654,0.277983,12233.78,0.029388,7,8.235095,6.963853,0.440712,54.711429
4,1,01869cd4,7,6.890234,8.668465,11.871820,10.638380,20.315555,12.972321,-0.649209,...,20.827719,1293,0.188759,11745.94,0.008367,7,8.492491,7.331352,0.445953,2.754127


## 5. Descriptor Correlations And Decision Signals

In [5]:
"Rank test-safe descriptors by association with well-level residual bias and residual magnitude."

candidate_descriptors = [
    "source_rows",
    "md_span",
    "z_span",
    "xy_span",
    "gr_missing_rate",
    "gr_mean",
    "gr_std",
    "known_tvt_rows",
    "known_tvt_fraction",
    "last_known_tvt_input",
    "recent_tvt_slope_last_50",
    "max_eval_progress",
    "mean_eval_progress",
    "max_distance_from_boundary_md",
    "mean_abs_projected_delta_10",
    "max_abs_projected_delta_10",
    "max_distance_log1p",
    "mean_distance_log1p",
]

targets = ["mean_residual", "abs_mean_residual", "rmse", "mae", "p90_abs_residual"]
correlation_rows = []
for descriptor in candidate_descriptors:
    for target in targets:
        subset = well_audit[[descriptor, target]].replace([np.inf, -np.inf], np.nan).dropna()
        if len(subset) < 20 or subset[descriptor].nunique() < 2 or subset[target].nunique() < 2:
            corr = np.nan
        else:
            corr = float(subset[descriptor].corr(subset[target], method="spearman"))
        correlation_rows.append(
            {
                "descriptor": descriptor,
                "target": target,
                "n_wells": int(len(subset)),
                "spearman": corr,
                "abs_spearman": abs(corr) if pd.notna(corr) else np.nan,
            }
        )

descriptor_correlations = pd.DataFrame(correlation_rows).sort_values(
    ["abs_spearman", "n_wells"], ascending=[False, False]
)

audit_findings = []
worst = well_audit.sort_values("hierarchy_priority_score", ascending=False).head(10)
max_abs_bias = float(well_audit["abs_mean_residual"].max())
p90_abs_bias = float(np.nanpercentile(well_audit["abs_mean_residual"], 90))
median_abs_bias = float(well_audit["abs_mean_residual"].median())
best_corr = descriptor_correlations.dropna(subset=["spearman"]).head(1)
if not best_corr.empty:
    row = best_corr.iloc[0]
    audit_findings.append(
        {
            "finding": "strongest_descriptor_signal",
            "value": f"{row['descriptor']} vs {row['target']} Spearman {row['spearman']:.3f} across {int(row['n_wells'])} wells",
            "modeling_implication": "Use this descriptor family first when testing contextual offsets.",
        }
    )
audit_findings.extend(
    [
        {
            "finding": "well_bias_distribution",
            "value": f"median abs well mean residual {median_abs_bias:.3f}; p90 {p90_abs_bias:.3f}; max {max_abs_bias:.3f}",
            "modeling_implication": "Persistent signed well bias is present if p90/max are materially above zero, but descriptor predictability determines whether hierarchy is useful.",
        },
        {
            "finding": "worst_wells_by_priority",
            "value": ", ".join(worst["wellname"].astype(str).head(5).tolist()),
            "modeling_implication": "Use these wells for residual plots and fold-level checks before adding flexible hierarchy.",
        },
    ]
)
findings = pd.DataFrame(audit_findings)

descriptor_correlations.head(12)

,descriptor,target,n_wells,spearman,abs_spearman
68,max_distance_from_boundary_md,mae,773,0.174534,0.174534
83,max_distance_log1p,mae,773,0.174534,0.174534
67,max_distance_from_boundary_md,rmse,773,0.170417,0.170417
82,max_distance_log1p,rmse,773,0.170417,0.170417
18,xy_span,mae,773,0.166100,0.166100
69,max_distance_from_boundary_md,p90_abs_residual,773,0.166058,0.166058
84,max_distance_log1p,p90_abs_residual,773,0.166058,0.166058
17,xy_span,rmse,773,0.164187,0.164187
3,source_rows,mae,773,0.163891,0.163891
8,md_span,mae,773,0.163891,0.163891


## 6. Export Artifacts

In [6]:
"Export the residual audit tables, concise report, and run bundle."

well_summary_path = ARTIFACT_TABLES / f"model__hierarchical_residual_bias_well_summary__{DATE_STAMP}.csv"
descriptor_corr_path = ARTIFACT_TABLES / f"model__hierarchical_residual_bias_descriptor_correlations__{DATE_STAMP}.csv"
fold_summary_path = ARTIFACT_TABLES / f"model__hierarchical_residual_bias_fold_summary__{DATE_STAMP}.csv"
findings_path = ARTIFACT_TABLES / f"model__hierarchical_residual_bias_findings__{DATE_STAMP}.csv"
report_path = ARTIFACT_REPORTS / f"model__hierarchical_residual_bias_audit__{DATE_STAMP}.md"
comparison_path = ARTIFACT_TABLES / f"model__hierarchical_residual_bias_comparison__{DATE_STAMP}.csv"

well_audit.sort_values("hierarchy_priority_score", ascending=False).to_csv(well_summary_path, index=False)
descriptor_correlations.to_csv(descriptor_corr_path, index=False)
fold_summary.to_csv(fold_summary_path, index=False)
findings.to_csv(findings_path, index=False)

comparison = pd.DataFrame(
    [
        {
            "date": DATE_STAMP,
            "model": "hierarchical_residual_bias_audit",
            "local_rmse": LOCAL_BASELINE_RMSE,
            "public_lb_reference": PUBLIC_BASELINE_RMSE,
            "decision": "diagnostic_only_continue_to_contextual_offsets",
            "notes": "Audit uses sampled out-of-fold residuals from huber_plus_hsgp_dps_m8; no new predictions submitted.",
        }
    ]
)
comparison.to_csv(comparison_path, index=False)

best_corr_rows = descriptor_correlations.dropna(subset=["spearman"]).head(5)

def markdown_table(frame):
    display_frame = frame.copy()
    for col in display_frame.select_dtypes(include=["float"]).columns:
        display_frame[col] = display_frame[col].map(lambda value: "" if pd.isna(value) else f"{value:.4f}")
    columns = list(display_frame.columns)
    header = "| " + " | ".join(columns) + " |"
    divider = "| " + " | ".join(["---"] * len(columns)) + " |"
    rows = []
    for _, row in display_frame.iterrows():
        rows.append("| " + " | ".join(str(row[col]) for col in columns) + " |")
    return "\n".join([header, divider] + rows)

best_corr_table = markdown_table(best_corr_rows[["descriptor", "target", "n_wells", "spearman"]])
fold_summary_table = markdown_table(fold_summary)
report_lines = [
    "# Hierarchical Residual Bias Audit",
    "",
    f"Date: {DATE_STAMP}",
    "",
    "## Purpose",
    "",
    "Audit whether the current Huber + additive Hilbert residual model leaves well-level residual structure that could justify contextual partial pooling.",
    "",
    "## Input",
    "",
    f"- Residual sample: `{RESIDUAL_PATH}`",
    f"- Candidate audited: `{TARGET_CANDIDATE}`",
    f"- Sampled residual rows: {len(residuals):,}",
    f"- Wells: {residuals['wellname'].nunique():,}",
    "",
    "## Key Metrics",
    "",
    f"- Current local diagnostic reference: **{LOCAL_BASELINE_RMSE:.4f}** RMSE",
    f"- Current public LB reference: **{PUBLIC_BASELINE_RMSE:.3f}** RMSE",
    f"- Median absolute well mean residual: **{median_abs_bias:.3f}** TVT units",
    f"- 90th percentile absolute well mean residual: **{p90_abs_bias:.3f}** TVT units",
    f"- Max absolute well mean residual: **{max_abs_bias:.3f}** TVT units",
    "",
    "## Strongest Descriptor Signals",
    "",
    best_corr_table,
    "",
    "## Fold Summary",
    "",
    fold_summary_table,
    "",
    "## Decision",
    "",
    "Continue to `01_empirical_bayes_offsets.ipynb`, but keep the first correction small and shrinkage-controlled. The audit shows enough well-level signed residual variation to test contextual offsets, while the descriptor correlations should guide which known-zone and geometry summaries enter first.",
    "",
    "## Artifacts",
    "",
    f"- Well summary: `{well_summary_path}`",
    f"- Descriptor correlations: `{descriptor_corr_path}`",
    f"- Fold summary: `{fold_summary_path}`",
    f"- Findings: `{findings_path}`",
    f"- Comparison row: `{comparison_path}`",
]
report = "\n".join(report_lines) + "\n"
report_path.write_text(report, encoding="utf-8")

for path in [well_summary_path, descriptor_corr_path, fold_summary_path, findings_path, comparison_path, report_path]:
    shutil.copy2(path, ARTIFACT_LOGS / path.name)

metadata = {
    "run_id": RUN_ID,
    "candidate": TARGET_CANDIDATE,
    "residual_path": str(RESIDUAL_PATH),
    "n_rows": int(len(residuals)),
    "n_wells": int(residuals["wellname"].nunique()),
    "local_reference_rmse": LOCAL_BASELINE_RMSE,
    "public_reference_rmse": PUBLIC_BASELINE_RMSE,
    "decision": "continue_to_contextual_offsets",
}
(ARTIFACT_LOGS / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(report_path)
print(well_summary_path)
print(descriptor_corr_path)
print(fold_summary_path)

C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\reports\model__hierarchical_residual_bias_audit__2026-06-22.md
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_residual_bias_well_summary__2026-06-22.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_residual_bias_descriptor_correlations__2026-06-22.csv
C:\Users\sardo\OneDrive\Desktop\Classes\Projects\Wellbore-tvt-Modeling\artifacts\tables\model__hierarchical_residual_bias_fold_summary__2026-06-22.csv


## Results

This notebook is diagnostic-only. It does not create a Kaggle submission; it decides whether the next hierarchical notebook should test contextual empirical Bayes offsets.